# Sequential Optimization for MOLLUSCA
## Traditional Categories & How MDO modules roughly fit
- Hydrodynamic Design (External Shape & Hull)
    - Geometry
- Power & Propulsion (Energy Systems)
    - Speed
    - Plastic
    - Power
- Structural & Mass Distribution
    - Mass
    - Stability
- Guidance, Navigation & Control
    - Manueverability
- Mission Planning
    - No module fits here
### Notes
- Cost doesn't fit anywhere

In [ ]:
clear all

In [ ]:
# Importing packages/math techniques used!
%reset -f
import openmdao
import numpy as np
import openmdao.api as om
from math import pi,sqrt,cos,sin,exp
from random import random
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
%matplotlib inline
print("OpenMDAO version:", openmdao.__version__)

In [ ]:
#Defining 2 types of constants

class Kspec: #these could theoretically change and impact performance but are being held constant
    C_d = 0.1    #---------------------------------------- Drag coefficient
    F_t = 40    #----------------------------------------- Thrust force [N]
    hotel_load = 41.76  #--------------------------------- Electronic load [W/hr]
    H = 0.0057    #--------------------------------------- Plate submersion height [m]
    plastic_concentration = 20 #-------------------------- Plastic density [particles/m^2]
    platewidth = 0.01    #-------------------------------- Plate thickness [m]
    h_b = 0.15 #----------==----------------------------- Total Electronic box thickness [m]
    h_bt = 0.005 #--------------------------------------- Electronic box thickness [m]
    amplitude = 0.005    #------------------------------- Height of undulation [m]
    wavelength = 0.5    #-------------------------------- Plate Wavelength [m]
    t_op = 5 #------------------------------------------- Duration of operation [hr]
    t_p = 0.005    #------------------------------------- Pontoon thickness [m]
    componentweight = 0   #------------------------------ Additional weight [kg]
    mass_motor1 = 2.29    #------------------------------ Motor mass (for plate) [kg]
    mass_motor2 = 1.22    #------------------------------ Motor mass (for mesh) [kg]
    mass_propellers = 0.7346   #------------------------- Propeller mass (1 propeller) [kg]
    t_l = 0.01    #-------------------------------------- Structure thickness [m]
    RPMmesh = 120    #----------------------------------- Gear RPM [rev/min]
    mu = 0.12    #--------------------------------------- Friction Coefficient
    Cf = 0.03    #--------------------------------------- Skin Friction Coefficient
    motoref = 0.75 #------------------------------------- Motor Efficiency
    batef = 0.9 #---------------------------------------- Battery Storage Efficiency
    solperday = 5627 #----------------------------------- Solar energy recieved per m^2 per day
    solcap = 0.18 #-------------------------------------- Solar Capacity
    startingtime = 7 #----------------------------------- Hour at which mission begins
    h1 = 0 #--------------------------------------------- Height at water surface
    percent_travel = 0.25 #------------------------------ Percent of Mission time spent traveling
        
class K: #these will not change (true constants)
    rho = 1000    #------------------------------------------------ Water density [kg/m^3]
    g = 9.81    #-------------------------------------------------- Gravitational acceleration [m/s^2]
    matpontoon = 1    #-------------------------------------------- Pontoon Material (1=HDPE, 2=LDPE)
    matstruct = 1    #--------------------------------------------- Structure Material: 1 fiberglass, 2 aluminum
    matplate = 1    #---------------------------------------------- Plate Material: 1 fiberglass, 2 aluminum
    cm_motorcontrol = 180    #------------------------------------- Cost of 2 motor controllers (Talon) [$]
    cm_powerdistboard = 94    #------------------------------------ Cost of power distribution board (Mersen) [$]
    cm_wifi_adapter = 24    #-------------------------------------- Cost of WiFi adapter (Detroit) [$]
    cm_solarchargecontrol = 159    #------------------------------- Cost of solar charge controller (EP) [$]
    cm_rcsys = 78    #--------------------------------------------- Cost of RC system (Flysky) [$]
    cm_adaIMU = 15    #-------------------------------------------- Cost of IMU (adafruit) [$]
    cm_adaGPS = 30    #-------------------------------------------- Cost of IMU (adafruit) [$]
    cm_controlarduino = 50    #------------------------------------ Cost of frontseat microcontroller (arduino) [$]
    cm_compraspberry = 50    #------------------------------------- Cost of backseat microcomputer (raspberry pi) [$]
    mass_gear = 2.25    #------------------------------------------ Mass of gear pulling mesh [kg]
    radius_gear = 0.05    #---------------------------------------- Radius of gear pulling mesh [m]
    b = 15     #--------------------------------------------------- Number of bars in plate
    S = 1.5    #--------------------------------------------------- Safety Factor for torque on 1 bar

In [ ]:
def initialization(x0,K,Kspec):
    L = x0[0]
    D = x0[1]
    W = x0[2]
    h = x0[3]
    B_Wh = x0[4]
    RPM = x0[5]
    w = x0[6]
    l = x0[7]
    wb = x0[8]
    lb = x0[9]
    Px = x0[10]
    wp = x0[11]
    lp = x0[12]
    v = x0[13]
#Geometry
    area_cylinderx0 = pi*(D/2)**2
    area_totalx0 = area_cylinderx0 + w*Kspec.platewidth
    volume_cylinderx0 = area_cylinderx0*L
    volume_cylinderintx0 = pi*((D-2*Kspec.t_p)/2)**2*(L-2*Kspec.t_p)
    volume_platex0 = w*l*Kspec.platewidth
    volume_totalx0 = volume_cylinderx0 + volume_platex0
    volume_boxx0 = Kspec.h_b*wb*lb
    volume_boxintx0 = (Kspec.h_b-2*Kspec.h_bt)*(wb-2*Kspec.h_bt)*(lb-2*Kspec.h_bt)
    volume_shaftx0= pi/4*(Kspec.platewidth)**2*l/Kspec.wavelength*np.sqrt((pi*Kspec.amplitude)**2+Kspec.wavelength**2)
    volume_structx0 = Kspec.t_l**2*(4*(h+1/2*Kspec.h_b+1/2*D)+2*(W-2*Kspec.t_l)+2*(W-D)+lb)
    volume= 2*volume_cylinderx0+volume_platex0+volume_boxx0+volume_structx0+volume_shaftx0
#Mass
    mass_panelx0 = 3.492*wp*lp
    mass_meshx0 = 0.002002*(Kspec.h1+h)*(W-D/2)
    mass_boxx0 = 920*(volume_boxx0-volume_boxintx0)
    mass_batteryx0 = B_Wh*0.008
    if K.matpontoon == 1: 
        rho_pontoon = 970 
    else:
        rho_pontoon = 940
    mass_pontoonx0 = rho_pontoon*(volume_cylinderx0-volume_cylinderintx0)
    if K.matstruct == 1:
        rho_struct = 2700
    else:
        rho_struct = 2588
    mass_structx0 = rho_struct*volume_structx0
    if K.matplate == 1:
        rho_plate = 2700
    else: rho_plate = 2588
    mass_platex0 = w*l*Kspec.platewidth*rho_plate
    mass_shaftx0 = rho_plate*volume_shaftx0
    robot_massx0 = mass_boxx0 +  mass_structx0 + mass_batteryx0 + Kspec.mass_motor1 + Kspec.mass_motor2 + mass_panelx0 + Kspec.componentweight \
            + 2*mass_pontoonx0 + 2*Kspec.mass_propellers +mass_platex0 + mass_shaftx0 + mass_meshx0
#Speed
    max_speedx0 = np.sqrt(Kspec.F_t / (0.5*K.rho*Kspec.C_d*area_totalx0))
    travel_speedx0 = min(max_speedx0,2)
    max_power_propulsionx0 = 0.5*K.rho*Kspec.C_d*area_totalx0*travel_speedx0**3
    collection_power_propulsionx0 = 0.5*K.rho*Kspec.C_d*area_totalx0*v**3
    energy_propulsionx0 = (max_power_propulsionx0*Kspec.percent_travel + collection_power_propulsionx0*(1-Kspec.percent_travel))* Kspec.t_op/Kspec.motoref
#Stability
    cogx0 = (((2*mass_pontoonx0 + 2*Kspec.mass_propellers)*Kspec.h1 + (mass_boxx0 +  mass_structx0 + mass_batteryx0 + Kspec.mass_motor1 + Kspec.mass_motor2 \
          + mass_panelx0 + Kspec.componentweight)*h + mass_meshx0/2*(h)+(mass_platex0+mass_shaftx0)*(-(Kspec.H+ .5*Kspec.platewidth))) / 
            ((2*mass_pontoonx0 + 2*Kspec.mass_propellers) + (mass_boxx0 +  mass_structx0 + mass_batteryx0 + Kspec.mass_motor1 + Kspec.mass_motor2 \
            + mass_panelx0 + Kspec.componentweight) +mass_meshx0+mass_platex0+mass_shaftx0))
    mass_wpontoonx0 = K.rho*volume_cylinderx0
    mass_wplatex0 = K.rho*volume_platex0
    mass_wshaftx0 = K.rho*volume_shaftx0
    CoBx0 =(-mass_wpontoonx0*(2*D/(3*pi)) - (mass_wplatex0 + mass_wshaftx0)*(Kspec.H+.5*Kspec.platewidth)) /(mass_wpontoonx0+mass_wplatex0+mass_wshaftx0)
    KBx0 = (Kspec.H+Kspec.platewidth)+CoBx0
    inertia_pitchx0 = D*L**3/6+w*l**3/12
    inertia_rollx0 = 2*(L*D**3/12+D*L*(D/2+W/2)**2)+l*w**3/12
    BM_pitchx0 = inertia_pitchx0 /volume_totalx0
    BM_rollx0 = inertia_rollx0 /volume_totalx0
    H_meta_pitchx0 = KBx0+BM_pitchx0-cogx0
    H_meta_rollx0 = KBx0+BM_rollx0-cogx0
    Tx0 = D/2
    Fbx0 = K.rho*K.g*volume_cylinderx0
#Maneuverability
    Bx0 = W+D
    Cbx0 = volume_totalx0/(L*Bx0*Tx0)
    mprix0 = robot_massx0/(0.5*K.rho*L**3)
    Yvx0 = -(1+0.4*Cbx0*Bx0/Tx0)*pi*(Tx0/L)**2
    Yrx0 = -(-0.5+2.2*Bx0/Tx0-0.08*Bx0/Tx0)*pi*(Tx0/L)**2
    Nvx0 = -(1/2+2.4*Tx0/L)*pi*(Tx0/L)
    Nrx0 = -(1/4+0.039*Bx0/Tx0-0.56*Bx0/L)*pi*(Tx0/L)**2
    turningdiamx0 = 2*L-2*Px*L
    VarLx0 = Nvx0*(Yrx0-mprix0)-Yvx0*Nrx0
    VarTx0 = turningdiamx0/L
#Plastic
    deltawidthx0 = (w/0.11)
    fx0=RPM/60
    fittedx0 = (-0.4441*v**2-0.0042*fx0**2-0.0241*v*fx0+0.7367*v+0.0554*fx0-0.0518)*10293*deltawidthx0*(Kspec.plastic_concentration/20000)
    particlestotalx0 = fittedx0*3600*Kspec.t_op*(1-Kspec.percent_travel)
    v_wx0 = Kspec.wavelength*RPM/60 
    torquemaxx0 = 0
    phi = pi/2
    Tflowx0 = 60/RPM
    for tscale in range (0,8):
        if tscale == 0:
            t = 0
        else: t = Tflowx0/tscale
        torquetotalx0 = 0
        for e in range (1,K.b+1):
            x = l*(e-1)/K.b
            omegax0 = 2*pi*RPM/60;
            thetax0= 2*pi*x/Kspec.wavelength-omegax0*t+phi
            Ax0 = 0.0219*v_wx0**2-0.6169*v_wx0+64.592
            if  v_wx0*1000 > 49:
                Tflowx0 = -0.0002*(v_wx0*1000-23)**2-0.0475*(v_wx0*1000-23)+2
            else: Tflowx0 = -0.001*(v_wx0*1000-49)**2-0.0148*(v_wx0*1000-49)+1 
            Qpxx0 = -Ax0*2*pi/Tflowx0* (2*pi*x/(Tflowx0*v_wx0)-2*pi/Tflowx0*t+phi-pi/2) *np.cos(2*pi*x/(Tflowx0*v_wx0)-2*pi/Tflowx0*t+phi-pi/2)
            Qpzx0 =  Ax0*2*pi/Tflowx0* (2*pi*x/(Tflowx0*v_wx0)-2*pi/Tflowx0*t+phi-pi/2) *np.sin(2*pi*x/(Tflowx0*v_wx0)-2*pi/Tflowx0*t+phi-pi/2)
            Fwxx0=Qpxx0*Kspec.H*w*l*K.rho/K.b
            Fwzx0=Qpzx0*Kspec.H*w*l*K.rho/K.b
            Ix0 = (2*mass_shaftx0-2*K.rho*volume_shaftx0+ mass_platex0-K.rho*l*w*Kspec.platewidth)*Kspec.amplitude**2
            alphax0 = RPM*pi/3
            Td_shaftx0 = pi*Kspec.amplitude**2*(l/K.b)*Kspec.Cf*K.rho*(omegax0*Kspec.amplitude)**2
            Fgx0= K.g/K.b*(2*mass_shaftx0-2*K.rho*volume_shaftx0+mass_platex0-K.rho*l*w*Kspec.platewidth)
            Ffx0= Kspec.mu*Fgx0
            torqueperbarx0 = (Kspec.amplitude*(abs(Fgx0*sin(thetax0))+abs(Ffx0*cos(thetax0))+abs(Fwxx0)+abs(Fwzx0*cos(thetax0)))+Td_shaftx0+Ix0*alphax0)*K.S #Assumes clockwise
            torquetotalx0 = torquetotalx0 + torqueperbarx0
        torquemaxx0 = max(torquemaxx0,torquetotalx0)
        power_plasticx0 = torquemaxx0*RPM*2*pi/60
        energy_plasticx0 = power_plasticx0*Kspec.t_op/Kspec.motoref*(1-Kspec.percent_travel)
#Power
    panel_areax0=wp*lp
    wattsx0 = Kspec.solperday*panel_areax0*Kspec.solcap
    energy_used_total_Whx0=(energy_propulsionx0/Kspec.batef+energy_plasticx0/Kspec.batef+Kspec.hotel_load*Kspec.t_op) #Wh
    energy_used_Wx0=energy_used_total_Whx0/Kspec.t_op
    battery_energyx0 = B_Wh #initial conditions
    energy_solarx0 = 0
    startingx0 = int(np.real_if_close((10*Kspec.startingtime)))
    endingx0   = int(np.real_if_close((10*Kspec.startingtime + 10*Kspec.t_op)))
    for time in range (startingx0, endingx0):
        tx0 = time / 10  # current time [h]
        gaussx0 = 685.4 * exp(-((tx0 - 11.58)/4.714)**2)  # W/m²
        delta_energyx0 = min(Kspec.batef*gaussx0*panel_areax0*Kspec.solcap, B_Wh - battery_energyx0)
        battery_energyx0 = battery_energyx0 + 1/10*delta_energyx0-1/10*energy_used_Wx0
    energy_balancex0 = battery_energyx0
#Cost
    if K.matpontoon == 1:
        ponpriceperdiameterx0 =  742.1*L
    else:
        ponpriceperdiameterx0 = 1791*L
    cm_pontoonx0 = D*ponpriceperdiameterx0
    cm_panelx0 = 1276*lp*wp
    cm_batteryx0 = 0.538*B_Wh 
    if K.matplate == 1:
        platunitprice = 1.9
    else:
        platunitprice = 1.8
    cm_platex0 = mass_platex0*platunitprice
    if K.matstruct == 1:
        fraunitprice = 1.9
    else:
        fraunitprice = 1.8
    cm_structx0 = mass_structx0*fraunitprice
    Tameshx0 = (1/2*K.mass_gear + mass_meshx0)*K.radius_gear**2*Kspec.RPMmesh*pi/3
    Tlmeshx0 = 1/np.sqrt(2)*K.radius_gear*mass_meshx0*K.g
    torquemeshx0 = 1.5*(Tameshx0 + Tlmeshx0)
    PeakRPMmesh = 1.2*Kspec.RPMmesh
    PeakRPMplatex0 = 1.2*RPM
    thrustlbfper = Kspec.F_t*0.453592
#Other
    dragx0 = 0.5*Kspec.C_d*K.rho*area_totalx0

    return {
        "area_cylinderx0": area_cylinderx0,
        "area_totalx0": area_totalx0 ,
        "volume_cylinderx0": volume_cylinderx0,
        "volume_cylinderintx0": volume_cylinderintx0,
        "volume_platex0": volume_platex0,
        "volume_totalx0":  volume_totalx0,
        "volume_boxx0": volume_boxx0,
        "volume_boxintx0": volume_boxintx0,
        "volume_shaftx0": volume_shaftx0,
        "volume_structx0": volume_structx0,
        "volume": volume,
        "mass_panelx0":  mass_panelx0,
        "mass_meshx0": mass_meshx0,
        "mass_boxx0": mass_boxx0,
        "mass_batteryx0": mass_batteryx0,
        "mass_pontoonx0": mass_pontoonx0, 
        "mass_structx0": mass_structx0,
        "mass_platex0": mass_platex0,
        "mass_shaftx0": mass_shaftx0,
        "robot_massx0": robot_massx0,
        "max_speedx0":  max_speedx0,
        "travel_speedx0": travel_speedx0,
        "max_power_propulsionx0": max_power_propulsionx0,
        "collection_power_propulsionx0": collection_power_propulsionx0,
        "energy_propulsionx0": energy_propulsionx0,
        "cogx0": cogx0,
        "mass_wpontoonx0": mass_wpontoonx0,
        "mass_wplatex0": mass_wplatex0,
        "mass_wshaftx0": mass_wshaftx0,
        "CoBx0": CoBx0, 
        "KBx0": KBx0,
        "inertia_pitchx0": inertia_pitchx0,
        "inertia_rollx0": inertia_rollx0, 
        "BM_pitchx0": BM_pitchx0,
        "BM_rollx0": BM_rollx0,
        "H_meta_pitchx0": H_meta_pitchx0, 
        "H_meta_rollx0": H_meta_rollx0,
        "Tx0": Tx0,
        "Fbx0": Fbx0,
        "Bx0": Bx0,
        "Cbx0": Cbx0, 
        "mprix0": mprix0,
        "Yvx0": Yvx0,
        "Yrx0": Yrx0,
        "Nvx0": Nvx0,
        "Nrx0": Nrx0,
        "turningdiamx0": turningdiamx0, 
        "VarLx0": VarLx0,
        "VarTx0": VarTx0,
        "deltawidthx0": deltawidthx0,
        "fx0": fx0,
        "fittedx0": fittedx0,
        "particlestotalx0":particlestotalx0,
        "v_wx0": v_wx0,
        "Tflowx0": Tflowx0,
        "torquemaxx0": torquemaxx0,
        "power_plasticx0": power_plasticx0, 
        "energy_plasticx0": energy_plasticx0,
        "panel_areax0": panel_areax0,
        "wattsx0": wattsx0 ,
        "energy_used_total_Whx0": energy_used_total_Whx0,
        "energy_used_Wx0": energy_used_Wx0,
        "battery_energyx0": battery_energyx0,
        "energy_balancex0": energy_balancex0, 
        "dragx0": dragx0
    }

In [ ]:
#Design Vector for Hydrodynamic Optimization
x0 = [0.8,0.2,0.2,0,250,300,0.1,0.2,0.1,0.2,0,2.5,0.5, 0.5]

L = x0[0]
D = x0[1]
W = x0[2]
h = x0[3]
B_Wh = x0[4]
RPM = x0[5]
w = x0[6]
l = x0[7]
wb = x0[8]
lb = x0[9]
Px = x0[10]
wp = x0[11]
lp = x0[12]
v = x0[13]

#Define design variables for Hydrodynamics Optimization
xh = [D,W,w]
#Initialize Hydrodynamics Optimization
hd = initialization(x0,K,Kspec)


In [ ]:
class hydrodynamics_module(om.ExplicitComponent):
    def initialize(self): #Call constants
        self.options.declare('K')
        self.options.declare('Kspec')
    def setup(self): #Inherit from explicit component class
    #Inputs:Design Variables
        self.add_input('D',val=xh[0])
        self.add_input('W',val=xh[1])
        self.add_input('w',val=xh[2])

    #Inputs:Parameters
        self.add_input('rho')
        self.add_input('platewidth')
        self.add_input('C_d')
    #Inputs: Guesses
        self.add_input('L',val=x0[0])
        self.add_input('volume_total', val=hd["volume_totalx0"])
        self.add_input('robot_mass', val = hd["robot_massx0"])

    #Outputs
        self.add_output('drag', val=hd["dragx0"])
        self.add_output('VarL', val=hd["VarLx0"])

    #So we can take derivatives
        self.declare_partials(of="*",wrt="*",method='fd')
        
    def compute(self,inputs,outputs): #inputs and outputs are dictionaries so we need to unpack
    #Unpack inputs
        D = inputs['D']
        W = inputs['W']
        w = inputs['w']
        L  = inputs['L']
        rho = inputs['rho']
        platewidth = inputs['platewidth']
        volume_total = inputs['volume_total']
        robot_mass = inputs['robot_mass']
        C_d = inputs['C_d']
        
    #Geometry Equations
        area_cylinder = pi*(D/2)**2
        area_total = area_cylinder + w*platewidth
        drag = 0.5*C_d*rho*area_total
        
        B = W+D
        T = D/2
        Cb = volume_total/(L*B*T)
        mpri = robot_mass/(0.5*rho*L**3)
        Yv = -(1+0.4*Cb*B/T)*pi*(T/L)**2
        Yr = -(-0.5+2.2*B/T-0.08*B/T)*pi*(T/L)**2
        Nv = -(1/2+2.4*T/L)*pi*(T/L)
        Nr = -(1/4+0.039*B/T-0.56*B/L)*pi*(T/L)**2
        VarL = Nv*(Yr-mpri)-Yv*Nr 
        
     #Pack outputs   
        outputs['drag'] = drag
        outputs['VarL'] = VarL

In [ ]:
class ConstraintHydroModule(om.ExplicitComponent):
    def initialize(self):
        self.options.declare('K') #Call constants
    def setup(self):
        self.add_input('D',val=xh[0])
        self.add_input('W',val=xh[1])
        self.add_input('w',val=xh[2])
        self.add_input('VarL', val=hd["VarLx0"])

        for i in range(3):
            self.add_output(f'c{i}', val=0.0)
            
        self.declare_partials(of='*', wrt='*', method='fd')
    
    def compute(self,inputs,outputs):
        #Plastic Optimization
        outputs['c0']  = (inputs['w'] - (inputs['W']-inputs['D']))
        outputs['c1']  = -inputs['VarL']
        outputs['c2']  = (inputs['VarL'] - 1)

In [ ]:
class ObjectiveHydroModule(om.ExplicitComponent):
    def setup(self):
        self.add_input('drag',val=hd["dragx0"])
        self.add_output('objective',val=hd["dragx0"])
        self.declare_partials(of ="*", wrt="*", method = 'fd')
    def compute(self,inputs,outputs):
        drag=inputs['drag']
        objective=drag
        outputs['objective']=objective

In [ ]:
def hydro_optimization(xh,x0,K,Kspec,hd):
    prob = om.Problem()
    model = prob.model
    model.add_subsystem('hydro',subsys=hydrodynamics_module(), promotes=['*'])
    model.add_subsystem('cons',subsys=ConstraintHydroModule(), promotes=['*'])
    model.add_subsystem('object', subsys=ObjectiveHydroModule(), promotes=['*'])

    model.add_design_var('D', lower=0.12, upper=1)
    model.add_design_var('W', lower=0.01, upper=2)
    model.add_design_var('w', lower=0.01, upper=2)

    model.set_input_defaults('rho', val=1000)
    model.set_input_defaults('C_d', val=0.1)
    model.set_input_defaults('platewidth', val=0.01)

    ivc = om.IndepVarComp()
    ivc.add_output('rho',val=K.rho)
    ivc.add_output('C_d',val=Kspec.C_d)
    ivc.add_output('platewidth',val=Kspec.platewidth)
    model.add_subsystem('inputs',ivc,promotes=['*'])

    for i in range(3):
        model.add_constraint(f'c{i}', upper=0.0)
    
    model.add_objective('objective')

    #Setup Optimizer
    prob.driver=om.ScipyOptimizeDriver(optimizer='SLSQP',tol=1e-6,disp=False)
    prob.model.approx_totals(method='cs') 
    prob.driver.options['maxiter'] = 20000

    #Create recorder
    recorder=om.SqliteRecorder("cases.sql")

    #Attach recorder to problem
    prob.add_recorder(recorder)
    #Attach recorder to driver
    prob.driver.add_recorder(recorder)
    prob.driver.recording_options['record_derivatives']=True
    prob.driver.recording_options['record_outputs']=True
    prob.driver.recording_options['includes']=['*']

    prob.setup()
    prob.set_val('D', xh[0])
    prob.set_val('W',xh[1])
    prob.set_val('w',xh[2])
    
    #Attach recorder to subsystem
    model.object.add_recorder(recorder)

    prob.set_solver_print(0)

    prob.run_driver()    
    prob.record("final_state")
    prob.cleanup()

    #instantiate casereader (reads .sql and .json files stored from run)
    cr = om.CaseReader(prob.get_outputs_dir() / "cases.sql") #This is needed to access data stored by all drivers
    case=cr.get_case('final_state')
    totals=case.derivatives

    for name, meta in prob.model.get_constraints().items():
        val = prob.get_val(name)
        print(f"{name}: {val}")

    #Solver Recorder
    #Instantiate your CaseReader
    cr = om.CaseReader(prob.get_outputs_dir() /"cases.sql")
    #Get list of cases, without displaying them
    solver_cases = cr. list_cases("root.object",out_stream=None)
    #Plot last half of iterations
    num_cases = len(solver_cases)
    sub_cases = num_cases
    iterations = np.arange(0, num_cases)

    # Plot the final convergence of the design
    objective_history = []

    for case_id in solver_cases[-sub_cases:]:
        case = cr.get_case(case_id)
        objective_history.append(-case['objective'])

    fig, ax1 = plt.subplots()

    ax1.plot(iterations, np.array(objective_history).flatten())
    ax1.set(ylabel='Drag', xlabel='Model Iteration',title='Solver History')
    ax1.grid()

    plt.show()

    # Get the final values
    print(f"Number of cases: {len(solver_cases)}")
    print("Final values:")
    case = cr.get_case(solver_cases[-1])
    print('  objective:', case['objective'])

    #To see model convergence (Cases recorded by driver)
    # List driver cases (do not recurse to system/solver cases)
    driver_cases = cr.list_cases('driver', recurse=False)
    last_case = cr.get_case(driver_cases[-1])
    objectives = last_case.get_objectives(scaled=False)
    design_vars = last_case.get_design_vars(scaled=False)
    constraints = last_case.get_constraints(scaled=False)

    return {
        "prob": prob,
        "case": case,
        "totals": totals,
        "design_vars": design_vars,
        "constraints": constraints
    
   }

In [ ]:
#Run Hydrodynamics Optimization
hydroresults = hydro_optimization(xh,x0,K,Kspec,hd)
print(hydroresults["design_vars"])
print(hydroresults["constraints"])
#Update Hydrodynamics Variables
xh = [hydroresults["design_vars"]['D'][0],hydroresults["design_vars"]['W'][0], hydroresults["design_vars"]['w'][0]]

In [ ]:
#Design Vector for Energy System Optimization
     #  L     D     W h B_Wh RPM    w   l  wb  lb Px wp  lp   v
x0 = [0.8,xh[0],xh[1],0,250,300,xh[2],0.2,0.1,0.2,0,2.5,0.5,0.5]
#Define design variables for Energy System Optimization
xe = [B_Wh,RPM,l,wp,lp,v]
#Initialize Energy System Optimization
es = initialization(x0,K,Kspec)

In [ ]:
class energysys_module (om.ExplicitComponent): #speed,plastic,power
    def initialize(self):
        self.options.declare('K') #Call constants
        self.options.declare('Kspec')
    def setup(self):
    #Inputs: Design Variables
        self.add_input('B_Wh',val=xe[0])
        self.add_input('RPM',val=xe[1])
        self.add_input('l',val=xe[2]) 
        self.add_input('wp',val=xe[3])
        self.add_input('lp',val=xe[4])
        self.add_input('v', val=xe[5]
     #Inputs: Others   
        self.add_input('w',val=x0[6])
        self.add_input('area_total', val=es["area_totalx0"])
    #Inputs: Guesses 
        self.add_input('mass_plate', val=es["mass_platex0"])
        self.add_input('mass_shaft', val=es["mass_shaftx0"])     
    #Inputs:Parameters
        self.add_input('t_op')
        self.add_input('motoref')
        self.add_input('percent_travel')
        self.add_input('C_d')
        self.add_input('F_t')
        self.add_input('H')
        self.add_input('plastic_concentration')
        self.add_input('amplitude')
        self.add_input('wavelength')
        self.add_input('platewidth')
        self.add_input('mu')
        self.add_input('Cf')
        self.add_input('hotel_load')
        self.add_input('startingtime')
        self.add_input('batef')
        self.add_input('solperday')
        self.add_input('solcap')
        self.add_input('S')
        self.add_input('rho')
        self.add_input('g')
        self.add_input('b')
        
    #Outputs
        self.add_output('max_speed',val=es["max_speedx0"])
        self.add_output('energy_propulsion', val =es["energy_propulsionx0"])
        self.add_output('max_power_propulsion', val = es["max_power_propulsionx0"])
        self.add_output('collection_power_propulsion', val = es["collection_power_propulsionx0"])
        self.add_output('particlestotal', val=es["particlestotalx0"]) 
        self.add_output('energy_plastic', val= es["energy_plasticx0"])
        self.add_output('power_plastic', val=es["power_plasticx0"]) 
        self.add_output('torquemax', val= es["torquemaxx0"])
        self.add_output('battery_energy',val=es["battery_energyx0"])
        self.add_output('energy_balance', val=es["energy_balancex0"])
        self.add_output('watts', val=es["wattsx0"])
        
    #So we can take derivatives 
        self.declare_partials(of ="*",wrt="*",method='fd')
        
    def compute(self,inputs,outputs): #inputs and outputs are dictionaries so we need to unpack
    #Unpack inputs
        B_Wh=inputs['B_Wh']
        wp=inputs['wp']
        lp=inputs['lp']
        RPM = inputs['RPM'].item()
        w = inputs['w'].item()
        l = inputs['l'].item()
        v          = inputs['v'].item()
        area_total = inputs['area_total'].item()
        C_d        = inputs['C_d'].item()
        F_t        = inputs ['F_t'].item()
        t_op       = inputs ['t_op'].item()
        motoref    = inputs ['motoref'].item()
        percent_travel = inputs ['percent_travel'].item()
        mass_plate = inputs['mass_plate']
        mass_shaft = inputs['mass_shaft']
        H = inputs['H']
        plastic_concentration = inputs['plastic_concentration']
        amplitude = inputs['amplitude']
        wavelength = inputs['wavelength'].item()
        platewidth = inputs['platewidth']
        mu = inputs['mu']
        Cf = inputs['Cf']
        hotel_load  = inputs['hotel_load']
        startingtime = inputs['startingtime']
        batef = inputs['batef']
        solperday = inputs['solperday']
        solcap = inputs['solcap']
        S = inputs['S']
        rho = inputs['rho']
        g = inputs['g']
        b = int(np.real(inputs['b']).item())

    #Geometry Equations
        volume_shaft= pi/4*(platewidth)**2*l/wavelength*np.sqrt((pi*amplitude)**2+wavelength**2)
        
    #Speed Equations
        max_speed = np.sqrt(F_t / (0.5*rho*C_d*area_total))
        travel_speed = min(max_speed, 2) #NEW
        max_power_propulsion = 0.5*rho*C_d*area_total*travel_speed**3
        collection_power_propulsion = 0.5*rho*C_d*area_total*v**3
        energy_propulsion = (max_power_propulsion*percent_travel+collection_power_propulsion*(1-percent_travel)) * t_op/motoref

    #Plastic Equations
        deltawidth = w/0.11
        f=RPM/60
        fitted = (-0.4441*v**2-0.0042*f**2-0.0241*v*f+0.7367*v+0.0554*f-0.0518)*10293*deltawidth*(plastic_concentration/20000)
        particlestotal = fitted*3600*t_op*(1-percent_travel)
        
        #Initialize
        v_w = wavelength*RPM/60
        Tflow = 60/RPM
        torquemax = 0
        phi = pi/2 #K
       
        for tscale in range (0,8):
            if tscale == 0:
                t = 0
            else: t = Tflow/tscale
                
            torquetotal = 0
            
            for e in range (1,b+1):
                x = l*(e-1)/b
                omega = 2*pi*RPM/60;
                theta= 2*pi*x/wavelength-omega*t+phi
            
                A = 0.0219*v_w**2-0.6169*v_w+64.592

                Qpx = -A*2*pi/Tflow* (2*pi*x/(Tflow*v_w)-2*pi/Tflow*t+phi-pi/2)*np.cos(2*pi*x/(Tflow*v_w)-2*pi/Tflow*t+phi-pi/2)
                Qpz = A*2*pi/Tflow* (2*pi*x/(Tflow*v_w)-2*pi/Tflow*t+phi-pi/2)*np.sin(2*pi*x/(Tflow*v_w)-2*pi/Tflow*t+phi-pi/2)
                
                Fwx=Qpx*H*w*l*rho/b #Assuming QAin=QAout
                Fwz=Qpz*H*w*l*rho/b
            
                I = (2*mass_shaft-2*rho*volume_shaft+ mass_plate-rho*l*w*platewidth)*amplitude**2/b
                alpha = RPM*pi/3
                
                Td_shaft = pi*amplitude**2*(l/b)*Cf*rho*(omega*amplitude)**2
            
                Fg= g/b*(2*mass_shaft-2*rho*volume_shaft+mass_plate-rho*l*w*platewidth)
                Ff= mu*Fg
                torqueperbar = (amplitude*(np.abs(Fg*np.sin(theta))+np.abs(Ff*np.cos(theta))+np.abs(Fwx)+np.abs(Fwz*np.cos(theta)))+Td_shaft+I*alpha)*S #Assumes clockwise
                torquetotal = torquetotal + torqueperbar

            torquemax = max(torquemax,torquetotal)
        power_plastic = torquemax*RPM*2*pi/60
        energy_plastic = power_plastic*t_op*(1-percent_travel)/motoref

    #Power Equations
        panel_area=wp*lp
        watts = solperday*panel_area*solcap
        energy_used_total_Wh=(energy_propulsion/batef+energy_plastic/batef+hotel_load*t_op) #Wh
        energy_used_W=energy_used_total_Wh/t_op
        
        #Time loop
        battery_energy = B_Wh #initial conditions

        starting = int(np.real_if_close((10*startingtime).item()))
        ending   = int(np.real_if_close((10*startingtime + 10*t_op).item()))

        for time in range (starting, ending):
        #Solar charging (every hour, limited by battery capacity)
            t=1/10*time
            gauss = 685.4*exp(-((t-11.58)/4.714)**2)
            delta_energy = min(batef*gauss*panel_area*solcap, B_Wh - battery_energy)
            battery_energy = battery_energy + 1/10*delta_energy-1/10*energy_used_W
        energy_balance = battery_energy

    #Pack outputs
        outputs['max_speed']=max_speed
        outputs['energy_propulsion']=energy_propulsion
        outputs['max_power_propulsion'] = max_power_propulsion
        outputs['collection_power_propulsion'] = collection_power_propulsion
        outputs['particlestotal']=particlestotal
        outputs['energy_plastic']=energy_plastic
        outputs['power_plastic']=power_plastic
        outputs['torquemax']=torquemax
        outputs['battery_energy']=battery_energy
        outputs['energy_balance']=energy_balance
        outputs['watts']=watts

In [ ]:
class ConstraintEnergySysModule(om.ExplicitComponent):
    def initialize(self):
        self.options.declare('K') #Call constants
    def setup(self):
        self.add_input('B_Wh',val=xe[0])
        self.add_input('energy_balance',val=es["energy_balancex0"])
        
        for i in range(1):
            self.add_output(f'c{i}', val=0.0)
            
        self.declare_partials(of='*', wrt='*', method='fd')
    
    def compute(self,inputs,outputs):
        outputs['c0']  = (-inputs['energy_balance']+0.25*inputs['B_Wh']) 

In [ ]:
class ObjectiveEnergySysModule(om.ExplicitComponent):
    def setup(self):
        self.add_input('particlestotal',val=es["particlestotalx0"])
        self.add_output('objective',val=es["particlestotalx0"])
        self.declare_partials(of ="*", wrt="*", method = 'fd')
    def compute(self,inputs,outputs):
        particlestotal=inputs['particlestotal']
        objective=-particlestotal/1000
        outputs['objective']=objective

In [ ]:
def energysys_optimization(xe,x0,K,Kspec,es):
    prob = om.Problem()
    model = prob.model
    model.add_subsystem('energsys',subsys=energysys_module(),promotes=['*'])
    model.add_subsystem('cons',subsys=ConstraintEnergySysModule(),promotes=['*'])
    model.add_subsystem('object', subsys=ObjectiveEnergySysModule(),promotes=['*'])
    
    model.add_design_var('B_Wh',lower=250,upper=1000)
    model.add_design_var('RPM', lower=0.1,upper=500)
    model.add_design_var('l',   lower=0.01,upper=2)
    model.add_design_var('wp',  lower=0,upper=2.5)
    model.add_design_var('lp',  lower=0,upper=2)
    model.add_design_var('v',   lower=0,upper=0.5)

    model.set_input_defaults('C_d', val=0.1)
    model.set_input_defaults('F_t', val=40)
    model.set_input_defaults('hotel_load', val=41.76)
    model.set_input_defaults('H', val=0.0057)
    model.set_input_defaults('plastic_concentration', val= 20)
    model.set_input_defaults('platewidth', val=0.01)
    model.set_input_defaults('amplitude', val=0.005)
    model.set_input_defaults('wavelength', val=0.5)
    model.set_input_defaults('t_op', val=5)
    model.set_input_defaults('mu', val=0.12)
    model.set_input_defaults('Cf', val=0.03)
    model.set_input_defaults('motoref', val=0.75)
    model.set_input_defaults('batef', val=0.9)
    model.set_input_defaults('solperday', val=5627)
    model.set_input_defaults('solcap', val=0.18)
    model.set_input_defaults('startingtime', val=7)
    model.set_input_defaults('percent_travel',val=0.25)
    model.set_input_defaults('S',val=1.5)
    model.set_input_defaults('rho',val=1000)
    model.set_input_defaults('g',val=9.8)
    model.set_input_defaults('b',val=15)
        
    ivc = om.IndepVarComp()
    ivc.add_output('C_d',val=Kspec.C_d)
    ivc.add_output('F_t',val=Kspec.F_t)
    ivc.add_output('hotel_load',val=Kspec.hotel_load)
    ivc.add_output('H', val=Kspec.H)
    ivc.add_output('plastic_concentration', val= Kspec.plastic_concentration)
    ivc.add_output('platewidth', val=Kspec.platewidth)
    ivc.add_output('amplitude', val=Kspec.amplitude)
    ivc.add_output('wavelength', val=Kspec.wavelength)
    ivc.add_output('t_op', val=Kspec.t_op)
    ivc.add_output('mu', val=Kspec.mu)
    ivc.add_output('Cf', val=Kspec.Cf)
    ivc.add_output('motoref', val=Kspec.motoref)
    ivc.add_output('batef', val=Kspec.batef)
    ivc.add_output('solperday', val=Kspec.solperday)
    ivc.add_output('solcap', val=Kspec.solcap)
    ivc.add_output('startingtime', val=Kspec.startingtime)
    ivc.add_output('percent_travel',val=Kspec.percent_travel)
    ivc.add_output('S',val=K.S)
    ivc.add_output('rho',val=K.rho)
    ivc.add_output('g',val=K.g)
    ivc.add_output('b',val=K.b)
    
    model.add_subsystem('inputs',ivc,promotes=['*'])

    
    for i in range(1):
        model.add_constraint(f'c{i}', upper=0.0)

    model.add_objective('objective')

    #Setup Optimizer
    prob.driver=om.ScipyOptimizeDriver(optimizer='SLSQP',tol=1e-40,disp=False)
    prob.model.approx_totals(method='cs') 
    prob.driver.options['maxiter'] = 20000

    #Create recorder
    recorder=om.SqliteRecorder("cases.sql")

    #Attach recorder to problem
    prob.add_recorder(recorder)
    #Attach recorder to driver
    prob.driver.add_recorder(recorder)
    prob.driver.recording_options['record_derivatives']=True
    prob.driver.recording_options['record_outputs']=True
    prob.driver.recording_options['includes']=['*']

    prob.setup()
    prob.set_val('B_Wh',xe[0])
    prob.set_val('RPM', xe[1])
    prob.set_val('l',   xe[2])
    prob.set_val('wp',  xe[3])
    prob.set_val('lp',  xe[4])
    prob.set_val('v',   xe[5])
    
    #Attach recorder to subsystem
    model.object.add_recorder(recorder)

    prob.set_solver_print(0)

    prob.run_driver()
    prob.record("final_state")
    prob.cleanup()

    #instantiate casereader (reads .sql and .json files stored from run)
    cr = om.CaseReader(prob.get_outputs_dir() / "cases.sql") #This is needed to access data stored by all drivers
    case=cr.get_case('final_state')
    totals=case.derivatives

    for name, meta in prob.model.get_constraints().items():
        val = prob.get_val(name)
        print(f"{name}: {val}")

    #Solver Recorder
    #Instantiate your CaseReader
    cr = om.CaseReader(prob.get_outputs_dir() /"cases.sql")
    #Get list of cases, without displaying them
    solver_cases = cr. list_cases("root.object",out_stream=None)
    #Plot last half of iterations
    num_cases = len(solver_cases)
    sub_cases = num_cases
    iterations = np.arange(0, num_cases)

    # Plot the final convergence of the design
    objective_history = []

    for case_id in solver_cases[-sub_cases:]:
        case = cr.get_case(case_id)
        objective_history.append(-case['objective'])

    fig, ax1 = plt.subplots()

    ax1.plot(iterations, np.array(objective_history).flatten())
    ax1.set(ylabel='Plastic Particles', xlabel='Model Iteration',title='Solver History')
    ax1.grid()

    plt.show()

    # Get the final values
    print(f"Number of cases: {len(solver_cases)}")
    print("Final values:")
    case = cr.get_case(solver_cases[-1])
    print('  objective:', case['objective'])

    #To see model convergence (Cases recorded by driver)
    # List driver cases (do not recurse to system/solver cases)
    driver_cases = cr.list_cases('driver', recurse=False)
    last_case = cr.get_case(driver_cases[-1])
    objectives = last_case.get_objectives(scaled=False)
    design_vars = last_case.get_design_vars(scaled=False)
    constraints = last_case.get_constraints(scaled=False)

    return {
        "prob": prob,
        "case": case,
        "totals": totals,
        "design_vars": design_vars,
        "constraints": constraints
    
   }

In [ ]:
#Run Energy System Optimization
esresults = energysys_optimization(xe,x0,K,Kspec,es)
print(esresults["design_vars"])
print(esresults["constraints"])
#Update Energy System Variables
xe = [esresults["design_vars"]['B_Wh'][0], esresults["design_vars"]['RPM'][0], esresults["design_vars"]['l'][0], esresults["design_vars"]['wp'][0],
      esresults["design_vars"]['lp'][0], esresults["design_vars"]['v'][0]]

In [ ]:
#Design Vector for Structure and Mass Distribution Optimization
     #  L     D     W h  B_Wh   RPM     w     l  wb  lb Px   wp    lp     v
x0 = [0.5,xh[0],xh[1],0,xe[0],xe[1],xh[2],xe[2],0.1,0.2,0,xe[3],xe[4],xe[5]]
#Define design variables for Structure and Mass Distribution Optimization
xs = [L,h,wb,lb]
#Initialize Structure and Mass Distribution Optimization
sm = initialization(x0,K,Kspec)

In [ ]:
class structmass_module (om.ExplicitComponent):
    def initialize(self):
        self.options.declare('K') #Call constants
        self.options.declare('Kspec')
    def setup(self): #Inherit from explicit component class
    #Inputs:Design Variables
        self.add_input('L', val=xs[0])
        self.add_input('h', val=xs[1])
        self.add_input('wb', val=xs[2])
        self.add_input('lb', val=xs[3])
    #Inputs:Others
        self.add_input('D', val=x0[1])
        self.add_input('W', val=x0[2])
        self.add_input('B_Wh', val=x0[4])
        self.add_input('w', val=x0[6])
        self.add_input('l', val=x0[7])
        self.add_input('wp',val=x0[11])
        self.add_input('lp',val=x0[12])
        
    #Inputs:Outputs from other modules
        self.add_input('volume_shaft', val=sm["volume_shaftx0"])
        self.add_input('area_cylinder', val=sm["area_cylinderx0"])
       
    #Inputs:Parameters
        self.add_input('platewidth')
        self.add_input('mass_motor1')
        self.add_input('mass_motor2')
        self.add_input('componentweight')
        self.add_input('mass_propellers')
        self.add_input('h1')
        self.add_input('H')
        self.add_input('t_p')
        self.add_input('h_b')
        self.add_input('h_bt')
        self.add_input('t_l')
        self.add_input('matpontoon')
        self.add_input('matstruct')
        self.add_input('matplate')
        self.add_input('rho')
        self.add_input('g')
        
    # Outputs 
        #MASS
        self.add_output('mass_shaft', val = sm["mass_shaftx0"])
        self.add_output('mass_plate', val = sm["mass_platex0"])
        self.add_output('mass_mesh',val = sm["mass_meshx0"])
        self.add_output('mass_struct',val = sm["mass_structx0"])
        self.add_output('mass_box',val = sm["mass_boxx0"]) 
        self.add_output('mass_battery',val = sm["mass_batteryx0"]) 
        self.add_output('mass_panel',val = sm["mass_panelx0"])
        self.add_output('mass_pontoon',val = sm["mass_pontoonx0"])
        #STABILITY
        self.add_output('H_meta_pitch', val= sm["H_meta_pitchx0"])
        self.add_output('H_meta_roll', val= sm["H_meta_rollx0"])
        self.add_output('volume_cylinder', val= sm["volume_cylinderx0"])
        self.add_output('inertia_roll', val= sm["inertia_rollx0"])

                        
    #So we can take derivatives  
        self.declare_partials(of ="*",wrt="*",method='fd')

        
    def compute(self,inputs,outputs):#inputs and outputs are dictionaries so we need to unpack
    #Unpack inputs
        L = inputs['L']
        D = inputs['D']
        W = inputs['W']
        h = inputs['h']
        B_Wh = inputs['B_Wh']
        w = inputs['w']
        l = inputs['l']
        wb = inputs['wb']
        lb = inputs['lb']
        wp = inputs['wp']
        lp = inputs['lp']
        
        volume_shaft = inputs['volume_shaft']
        area_cylinder = inputs['area_cylinder']
        platewidth = inputs['platewidth']
        mass_motor1 = inputs['mass_motor1']
        mass_motor2 = inputs['mass_motor2']
        componentweight = inputs['componentweight']
        mass_propellers = inputs['mass_propellers']
        h1 = inputs['h1']
        H = inputs['H']
        t_p = inputs['t_p']
        h_b = inputs['h_b']
        h_bt = inputs['h_bt']
        t_l  = inputs['t_l']
        matpontoon = inputs['matpontoon']
        matstruct = inputs['matstruct']
        matplate = inputs['matplate']
        rho = inputs['rho']
        g = inputs['g']

    #Geometry Equations
        volume_cylinder = area_cylinder*L
        volume_cylinderint = pi*((D-2*t_p)/2)**2*(L-2*t_p)
        volume_box = h_b*wb*lb
        volume_boxint = (h_b-2*h_bt)*(wb-2*h_bt)*(lb-2*h_bt)
        volume_struct = t_l**2*(4*(h+1/2*h_b+1/2*D)+2*(W-2*t_l)+2*(W-D)+lb)
        volume_plate = w*l*platewidth
        volume_total = volume_cylinder + volume_plate
    #Mass Equations
        mass_panel = 3.492*wp*lp
        mass_mesh = 0.002002*(h1+h)*(W-D/2)
        mass_box = 920*(volume_box-volume_boxint)
        mass_battery = B_Wh*0.008
        #Decision               
        if matpontoon == 1: #HDPE
            rho_pontoon = 970
        else:
            rho_pontoon = 940 #LDPE
                    
        mass_pontoon = rho_pontoon*(volume_cylinder-volume_cylinderint)
        #Decision
        if matstruct == 1: #aluminum
            rho_struct = 2700
        else:
            rho_struct = 2588 #fiberglass

        mass_struct = rho_struct*volume_struct
        #Decision
        if matplate == 1: #aluminum
            rho_plate = 2700
        else:
            rho_plate = 2588 #fiberglass

        mass_plate = w*l*platewidth*rho_plate
        mass_shaft = rho_plate*volume_shaft
        
        #Stability Equations
        cog = (((2*mass_pontoon + 2*mass_propellers)*h1 + (mass_box +  mass_struct + mass_battery + mass_motor1 + mass_motor2 + mass_panel \
                 + componentweight)*h + mass_mesh/2*(h)+(mass_plate+mass_shaft)*(-(H+ .5*platewidth))) / (2*mass_pontoon + 2*mass_propellers \
                 + mass_box +  mass_struct + mass_battery + mass_motor1 + mass_motor2 + mass_panel + componentweight +mass_mesh+mass_plate+mass_shaft))
   
        mass_wpontoon = rho*volume_cylinder
        mass_wplate = rho*volume_plate
        mass_wshaft = rho*volume_shaft
        CoB =(-mass_wpontoon*(2*D/(3*pi)) - (mass_wplate + mass_wshaft)*(H+.5*platewidth)) / (mass_wpontoon+mass_wplate+mass_wshaft)
        KB = (H+platewidth)+CoB

        inertia_pitch = D*L**3/6+w*l**3/12
        inertia_roll = 2*(L*D**3/12+D*L*(D/2+W/2)**2)+l*w**3/12

        BM_pitch = inertia_pitch /volume_total
        BM_roll = inertia_roll /volume_total
    
        H_meta_pitch = KB+BM_pitch-cog
        H_meta_roll = KB+BM_roll-cog
        Fb = rho*g*volume_cylinder
    #Pack outputs
        outputs['mass_shaft'] = mass_shaft
        outputs['mass_plate'] = mass_plate
        outputs['mass_mesh'] = mass_mesh
        outputs['mass_struct'] = mass_struct
        outputs['mass_box'] = mass_box
        outputs['mass_battery']=mass_battery
        outputs['mass_panel']=mass_panel
        outputs['mass_pontoon']=mass_pontoon
        outputs['H_meta_pitch']= H_meta_pitch
        outputs['H_meta_roll'] = H_meta_roll
        outputs['volume_cylinder'] = volume_cylinder
        outputs['inertia_roll'] = inertia_roll

In [ ]:
class ConstraintSMModule(om.ExplicitComponent):
    def initialize(self):
        self.options.declare('K') #Call constants
    def setup(self):

        self.add_input('H_meta_pitch',val= sm["H_meta_pitchx0"])
        self.add_input('H_meta_roll',val= sm["H_meta_rollx0"])
        self.add_input('g')
        ###########
        self.add_input('rho')
        self.add_input('volume_cylinder')
        self.add_input('mass_box',val= sm["mass_boxx0"])
        self.add_input('mass_struct',val= sm["mass_structx0"])
        self.add_input('mass_battery',val= sm["mass_batteryx0"])
        self.add_input('mass_motor1')
        self.add_input('mass_motor2')
        self.add_input('mass_panel',val= sm["mass_panelx0"])
        self.add_input('componentweight')
        self.add_input('mass_pontoon')
        self.add_input('mass_propellers')
        self.add_input('mass_plate',val= sm["mass_platex0"])
        self.add_input('mass_shaft',val= sm["mass_shaftx0"])
        self.add_input('mass_mesh',val= sm["mass_meshx0"])

        for i in range(3):
            self.add_output(f'c{i}', val=0.0)
            
        self.declare_partials(of='*', wrt='*', method='fd')
    
    def compute(self,inputs,outputs):
        mass_box = inputs['mass_box']
        mass_struct =inputs['mass_struct']
        mass_battery =inputs['mass_battery']
        mass_motor1 =inputs['mass_motor1']
        mass_motor2 =inputs['mass_motor2']
        mass_panel =inputs['mass_panel']
        componentweight = inputs['componentweight']
        mass_pontoon =inputs['mass_pontoon']
        mass_propellers = inputs['mass_propellers']
        mass_plate = inputs['mass_plate']
        mass_shaft = inputs['mass_shaft']
        mass_mesh = inputs['mass_mesh']        

        robot_mass = mass_box +  mass_struct + mass_battery + mass_motor1 + mass_motor2 + mass_panel + componentweight + 2*mass_pontoon + 2*mass_propellers +mass_plate + mass_shaft + mass_mesh

        
        #Pack Outputs
        outputs['c0']  = (0.1 - inputs['H_meta_pitch'])
        outputs['c1']  = (0.1 - inputs['H_meta_roll'])
        outputs['c2']  = (inputs['g']*robot_mass - inputs['g']*inputs['rho']*inputs['volume_cylinder']+9.8)     # VarT ≤ 3

In [ ]:
class ObjectiveSMModule(om.ExplicitComponent):
    def setup(self):
        self.add_input('mass_box',val= sm["mass_boxx0"])
        self.add_input('mass_struct',val= sm["mass_structx0"])
        self.add_input('mass_battery',val= sm["mass_batteryx0"])
        self.add_input('mass_motor1')
        self.add_input('mass_motor2')
        self.add_input('mass_panel',val= sm["mass_panelx0"])
        self.add_input('componentweight')
        self.add_input('mass_pontoon')
        self.add_input('mass_propellers')
        self.add_input('mass_plate',val= sm["mass_platex0"])
        self.add_input('mass_shaft',val= sm["mass_shaftx0"])
        self.add_input('mass_mesh',val= sm["mass_meshx0"])
    
        self.add_output('objective', val= sm["robot_massx0"])
        self.declare_partials(of ='*', wrt='*', method = 'fd')
    def compute(self,inputs,outputs):
        mass_box = inputs['mass_box']
        mass_struct =inputs['mass_struct']
        mass_battery =inputs['mass_battery']
        mass_motor1 =inputs['mass_motor1']
        mass_motor2 =inputs['mass_motor2']
        mass_panel =inputs['mass_panel']
        componentweight = inputs['componentweight']
        mass_pontoon =inputs['mass_pontoon']
        mass_propellers = inputs['mass_propellers']
        mass_plate = inputs['mass_plate']
        mass_shaft = inputs['mass_shaft']
        mass_mesh = inputs['mass_mesh']        

        objective = mass_box +  mass_struct + mass_battery + mass_motor1 + mass_motor2 + mass_panel + componentweight + 2*mass_pontoon + 2*mass_propellers +mass_plate + mass_shaft + mass_mesh
        outputs['objective']=objective

In [ ]:
def SM_optimization(xs,x0,K,Kspec):
    prob = om.Problem()
    model = prob.model
    model.add_subsystem('sm',subsys=structmass_module(), promotes=['*'])
    model.add_subsystem('object', subsys=ObjectiveSMModule(), promotes=['*'])
    model.add_subsystem('cons',subsys=ConstraintSMModule(), promotes=['*'])

    model.add_design_var('L', lower=0.01, upper=3)
    model.add_design_var('h', lower=0, upper=1)
    model.add_design_var('wb', lower=0.01, upper=1)
    model.add_design_var('lb', lower=0.01, upper=1)
    
    model.set_input_defaults('H', val=0.0057)
    model.set_input_defaults('platewidth', val=0.01)
    model.set_input_defaults('h_b', val=0.15)
    model.set_input_defaults('h_bt', val=0.005)
    model.set_input_defaults('t_p', val=0.005)
    model.set_input_defaults('componentweight', val=0)
    model.set_input_defaults('mass_motor1', val=2.29)
    model.set_input_defaults('mass_motor2', val=1.22)
    model.set_input_defaults('mass_propellers', val=0.7346)
    model.set_input_defaults('t_l', val=0.01)
    model.set_input_defaults('h1', val=0)
    model.set_input_defaults('g',val=9.81)
    model.set_input_defaults('matpontoon',val=1)
    model.set_input_defaults('matstruct',val=1)
    model.set_input_defaults('matplate',val= 1)
    model.set_input_defaults('rho',val=1000)

    ivc = om.IndepVarComp()
    ivc.add_output('H', val=Kspec.H)    
    ivc.add_output('platewidth', val=Kspec.platewidth)
    ivc.add_output('h_b', val=Kspec.h_b)
    ivc.add_output('h_bt', val=Kspec.h_bt)
    ivc.add_output('t_p', val=Kspec.t_p)
    ivc.add_output('componentweight', val=Kspec.componentweight)
    ivc.add_output('mass_motor1', val=Kspec.mass_motor1)
    ivc.add_output('mass_motor2', val=Kspec.mass_motor2)
    ivc.add_output('mass_propellers', val=Kspec.mass_propellers)
    ivc.add_output('t_l', val=Kspec.t_l)
    ivc.add_output('h1', val=Kspec.h1)
    ivc.add_output('g', val=K.g)
    ivc.add_output('matpontoon', val=K.matpontoon)
    ivc.add_output('matstruct', val=K.matstruct)
    ivc.add_output('matplate', val=K.matplate)
    ivc.add_output('rho', val=K.rho)

    model.add_subsystem('inputs',ivc,promotes=['*'])
    
    for i in range(3):
        model.add_constraint(f'c{i}', upper=0.0)

    model.add_objective('objective')

    #Setup Optimizer
    prob.driver=om.ScipyOptimizeDriver(optimizer='SLSQP',tol=1e-6,disp=False)
    prob.model.approx_totals(method='cs')
    prob.driver.options['maxiter'] = 20000

    #Create recorder
    recorder=om.SqliteRecorder("cases.sql")

    #Attach recorder to problem
    prob.add_recorder(recorder)
    #Attach recorder to driver
    prob.driver.add_recorder(recorder)
    prob.driver.recording_options['record_derivatives']=True
    prob.driver.recording_options['record_outputs']=True
    prob.driver.recording_options['includes']=['*']

    prob.setup()
    prob.set_val('L', xs[0])
    prob.set_val('h', xs[1])
    prob.set_val('wb', xs[2])
    prob.set_val('lb', xs[3])
    
    #Attach recorder to subsystem
    model.object.add_recorder(recorder)

    prob.set_solver_print(0)

    prob.run_driver()    
    prob.record("final_state")
    prob.cleanup()

    #instantiate casereader (reads .sql and .json files stored from run)
    cr = om.CaseReader(prob.get_outputs_dir() / "cases.sql") #This is needed to access data stored by all drivers
    case=cr.get_case('final_state')
    totals=case.derivatives

    for name, meta in prob.model.get_constraints().items():
        val = prob.get_val(name)
        print(f"{name}: {val}")

    #Solver Recorder
    #Instantiate your CaseReader
    cr = om.CaseReader(prob.get_outputs_dir() /"cases.sql")
    #Get list of cases, without displaying them
    solver_cases = cr. list_cases("root.object",out_stream=None)
    #Plot last half of iterations
    num_cases = len(solver_cases)
    sub_cases = num_cases
    iterations = np.arange(0, num_cases)

    # Plot the final convergence of the desing
    objective_history = []

    for case_id in solver_cases[-sub_cases:]:
        case = cr.get_case(case_id)
        objective_history.append(-case['objective'])

    fig, ax1 = plt.subplots()

    ax1.plot(iterations, np.array(objective_history).flatten())
    ax1.set(ylabel='Mass', xlabel='Model Iteration',title='Solver History')
    ax1.grid()

    plt.show()

    # Get the final values
    print(f"Number of cases: {len(solver_cases)}")
    print("Final values:")
    case = cr.get_case(solver_cases[-1])
    print('  objective:', case['objective'])

    #To see model convergence (Cases recorded by driver)
    # List driver cases (do not recurse to system/solver cases)
    driver_cases = cr.list_cases('driver', recurse=False)
    last_case = cr.get_case(driver_cases[-1])
    objectives = last_case.get_objectives(scaled=False)
    design_vars = last_case.get_design_vars(scaled=False)
    constraints = last_case.get_constraints(scaled=False)

    return {
        "prob": prob,
        "case": case,
        "totals": totals,
        "objectives": objectives,
        "design_vars": design_vars,
        "constraints": constraints
    
   }

In [ ]:
#Run Structure and Mass Distribution Optimization
smresults = SM_optimization(xs,x0,K,Kspec)
print(smresults["design_vars"])
print(smresults["constraints"])
#Update Structure and Mass Distribution Variables
xs = [smresults["design_vars"]['L'][0], smresults["design_vars"]['h'][0], smresults["design_vars"]['wb'][0], smresults["design_vars"]['lb'][0]]

In [ ]:
#Design Vector for Guidance, Navigation, and Control Optimization
     #    L     D     W     h  B_Wh   RPM     w     l    wb    lb Px   wp    lp     v
x0 = [xs[0],xh[0],xh[1],xs[1],xe[0],xe[1],xh[2],xe[2],xs[2],xs[3],0,xe[3],xe[4],xe[5]]
#Define design variables for Guidance, Navigation, and Control Optimization
xg = [Px]
#Initialize Guidance, Navigation, and Control Optimization
gnc = initialization(x0,K,Kspec)

In [ ]:
class GNC_module(om.ExplicitComponent):
    def initialize(self):
        self.options.declare('K') #Call constants
        self.options.declare('Kspec')
    def setup(self):
        #Inputs: Design Variables
        self.add_input('Px',val=x0[10])
        #Inputs: Others
        self.add_input('L', val=x0[0])
        #Outputs
        self.add_output('VarT',val= gnc["VarTx0"])
        #So we can take derivatives 
        self.declare_partials(of ="*",wrt="*",method='fd')
        
    def compute (self, inputs, outputs):#inputs and outputs are dictionaries so we need to unpack
    #Unpack inputs
        L  = inputs['L']
        Px = inputs['Px']
        
    #Maneuverability Equations
        turningdiam = 2*L-2*Px*L
        VarT = turningdiam/L

    #Pack outputs
        outputs['VarT'] = VarT

In [ ]:
class ObjectiveGNCModule(om.ExplicitComponent):
    def setup(self):
        self.add_input('VarT', val= gnc["VarTx0"])
        self.add_output('objective', val= gnc["VarTx0"])
        self.declare_partials(of='*', wrt='*', method='fd')
    def compute(self,inputs,outputs):
        VarT=inputs['VarT']
        objective=VarT
        outputs['objective']=objective

In [ ]:
def GNC_optimization(xg,x0,K,Kspec):
    prob = om.Problem()
    model = prob.model
    model.add_subsystem('GNC',subsys=GNC_module(), promotes=['*'])
    model.add_subsystem('object', subsys=ObjectiveGNCModule(), promotes=['*'])

    model.add_design_var('Px',lower=0,upper=0.5)
    
    for i in range(0):
        model.add_constraint(f'c{i}', upper=0.0)

    model.add_objective('objective')

    #Setup Optimizer
    prob.driver=om.ScipyOptimizeDriver(optimizer='SLSQP',tol=1e-6,disp=False)
    prob.model.approx_totals(method='cs')
    prob.driver.options['maxiter'] = 20000
    
    #Create recorder
    recorder=om.SqliteRecorder("cases.sql")

    #Attach recorder to problem
    prob.add_recorder(recorder)
    #Attach recorder to driver
    prob.driver.add_recorder(recorder)
    prob.driver.recording_options['record_derivatives']=True
    prob.driver.recording_options['record_outputs']=True
    prob.driver.recording_options['includes']=['*']

    prob.setup()
    prob.set_val('Px', xg[0])
    
    #Attach recorder to subsystem
    model.object.add_recorder(recorder)

    prob.set_solver_print(0)

    prob.run_driver()    
    prob.record("final_state")
    prob.cleanup()

    #instantiate casereader (reads .sql and .json files stored from run)
    cr = om.CaseReader(prob.get_outputs_dir() / "cases.sql") #This is needed to access data stored by all drivers
    case=cr.get_case('final_state')
    totals=case.derivatives

    for name, meta in prob.model.get_constraints().items():
        val = prob.get_val(name)
        print(f"{name}: {val}")

    #Solver Recorder
    #Instantiate your CaseReader
    cr = om.CaseReader(prob.get_outputs_dir() /"cases.sql")
    #Get list of cases, without displaying them
    solver_cases = cr. list_cases("root.object",out_stream=None)
    #Plot last half of iterations
    num_cases = len(solver_cases)
    sub_cases = num_cases
    iterations = np.arange(0, num_cases)

    # Plot the final convergence of the desing
    objective_history = []

    for case_id in solver_cases[-sub_cases:]:
        case = cr.get_case(case_id)
        objective_history.append(-case['objective'])

    fig, ax1 = plt.subplots()

    ax1.plot(iterations, np.array(objective_history).flatten())
    ax1.set(ylabel='Diameter of Relative Reversability', xlabel='Model Iteration',title='Solver History')
    ax1.grid()

    plt.show()

    # Get the final values
    print(f"Number of cases: {len(solver_cases)}")
    print("Final values:")
    case = cr.get_case(solver_cases[-1])
    print('  objective:', case['objective'])

    #To see model convergence (Cases recorded by driver)
    # List driver cases (do not recurse to system/solver cases)
    driver_cases = cr.list_cases('driver', recurse=False)
    last_case = cr.get_case(driver_cases[-1])
    objectives = last_case.get_objectives(scaled=False)
    design_vars = last_case.get_design_vars(scaled=False)
    constraints = last_case.get_constraints(scaled=False)

    return {
        "prob": prob,
        "case": case,
        "totals": totals,
        "objectives": objectives,
        "design_vars": design_vars,
        "constraints": constraints
    
   }

In [ ]:
#Run Guidance, Navigation, and Control Optimization
gncresults = GNC_optimization(xg,x0,K,Kspec)
print(gncresults["design_vars"])
print(gncresults["constraints"])
#Update Guidance, Navigation, and Control Variables
xg = [gncresults["design_vars"]['Px'][0]]

In [ ]:
#Final Design!
x0 = [xs[0],xh[0],xh[1],xs[1],xe[0],xe[1],xh[2],xe[2],xs[2],xs[3],xg[0],xe[3],xe[4],xe[5]]
FinalCharacteristics = initialization(x0,K,Kspec)
L = x0[0]
D = x0[1]
W = x0[2]
h = x0[3]
B_Wh = x0[4]
RPM = x0[5]
w = x0[6]
l = x0[7]
wb = x0[8]
lb = x0[9]
Px = x0[10]
wp = x0[11]
lp = x0[12]
v = x0[13]
print("H_C0: (w - (W-D))=", w-(W-D))
print("H_C1: -VarL =", - check["VarLx0"])
print("H_C2: (VarL - 1)=", check["VarLx0"]-1)
print("ES_C0: (-energy_balance+0.25*B_Wh)=",-check["energy_balancex0"]+0.25*B_Wh)
print("SM_C0:(0.1 - H_meta_pitch)=",0.1-check["H_meta_pitchx0"])
print("SM_C1:((0.1 - H_meta_roll)=",0.1-check["H_meta_rollx0"])
print("SM_C2: (g*robot_mass - Fb-9.8)=", K.g*check["robot_massx0"]-check["Fbx0"]+9.8)
print("%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
print("H_Obj: Min drag=", check["dragx0"])
print("ES_Obj: Max particlestotal=",check["particlestotalx0"])
print("SM_Obj: Min robotmass=",check["robot_massx0"])
print("GNC_Obj: Min VarT=",check["VarTx0"])